# 01 - Kết quả Giải đấu Vòng tròn (Round-Robin Tournament)

Phụ trách: **Người 1**. Notebook này **chỉ đọc log thô** từ `results/logs/round_robin_v1/` và `results/logs/profile_cost.json` (không nhập tay số liệu) để tạo các bảng thống kê và biểu đồ cho Chương 4 của báo cáo.

### Các nội dung phân tích chính:
1. **Ma trận đối đầu (`score_matrix`) & Bảng xếp hạng tổng hợp** ở 2 chế độ ngân sách (`time_per_move: 0.5s` và `default_config`).
2. **Thống kê Thắng/Hòa/Thua & Khoảng tin cậy 95% (Wilson CI & Pair-Bootstrap CI)** — Trả lời *Câu hỏi nghiên cứu 1* (MCTS vs Alpha-Beta) và *Câu hỏi nghiên cứu 3* (RL vs NN Supervised).
3. **Chi phí tính toán, Độ trễ (`mean`, `p50`, `p95`, `over_budget`) & Bộ nhớ RAM**.
4. **Phân tích lợi thế đi trước (Quân Đen vs Quân Trắng)** trên 6.000 ván đấu.
5. **Trích xuất ván đấu minh họa** theo quy tắc định trước từ `games.jsonl`.

In [ ]:
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO))
from eval.metrics import wilson_interval

LOGS = Path(os.environ.get("LOGS_DIR", REPO / "results" / "logs"))
FIGS = Path(os.environ.get("FIG_DIR", REPO / "results" / "figures"))
FIGS.mkdir(parents=True, exist_ok=True)

RUN_NAME = "round_robin_v1"
RUN_DIR = LOGS / RUN_NAME
SUMMARY_PATH = RUN_DIR / "summary.json"
GAMES_PATH = RUN_DIR / "games.jsonl"
PROFILE_PATH = LOGS / "profile_cost.json"

with open(SUMMARY_PATH, encoding="utf-8") as f:
    summary = json.load(f)

meta = summary["meta"]
agents = summary["config"]["agents"]
modes = summary["modes"]

LABELS = {
    "random": "Random (C1)",
    "heuristic": "Heuristic (C2)",
    "alphabeta": "Alpha-Beta (C3)",
    "mcts": "MCTS (C4)",
    "nn": "NN Sup (C5)",
    "rl": "RL Self-Play (C6)",
}

print(f"Run name      : {summary['run_name']}")
print(f"Total games   : {meta['total_games']:,} games ({summary['config']['games_per_pair']} games/pair, seeds={summary['config']['seeds']})")
print(f"Elapsed time  : {meta['elapsed_seconds']:.1f}s ({meta['elapsed_seconds']/3600:.2f} hours) with workers={meta['workers']}")
print(f"Platform / CPU: {meta['platform']} | {meta['processor']} ({meta['cpu_count']} logical cores)")
print(f"Git commit    : {meta['git_commit']}")

## 1. Bảng xếp hạng tổng thể & Ma trận đối đầu (`score_matrix`)

Điểm số chuẩn hóa của tác tử $A$ trước đối thủ $B$ được tính theo công thức của đề bài:
$$\text{score}(A, B) = \frac{\text{Wins}_A + 0.5 \times \text{Draws}}{\text{Total Games}}$$

In [ ]:
def build_leaderboard(mode_data: dict, agent_list: list) -> pd.DataFrame:
    totals = {a: {"wins": 0, "draws": 0, "losses": 0, "games": 0} for a in agent_list}
    for m in mode_data["matches"]:
        a, b = m["a"], m["b"]
        s = m["a_summary"]
        totals[a]["wins"] += s["wins"]
        totals[a]["draws"] += s["draws"]
        totals[a]["losses"] += s["losses"]
        totals[a]["games"] += s["games"]
        totals[b]["wins"] += s["losses"]
        totals[b]["draws"] += s["draws"]
        totals[b]["losses"] += s["wins"]
        totals[b]["games"] += s["games"]

    rows = []
    for a in agent_list:
        t = totals[a]
        n = t["games"]
        pts = t["wins"] + 0.5 * t["draws"]
        lo, hi = wilson_interval(pts, n)
        rows.append({
            "Agent": LABELS.get(a, a),
            "key": a,
            "Games": n,
            "Wins": t["wins"],
            "Draws": t["draws"],
            "Losses": t["losses"],
            "Win Rate (%)": round(100 * t["wins"] / n, 2),
            "Mean Score": round(mode_data["mean_score"][a], 4),
            "95% Wilson CI": f"[{lo:.3f}, {hi:.3f}]",
            "ci_lo": lo,
            "ci_hi": hi,
            "Over-budget Moves": mode_data["budget"]["by_agent"][a]["over_budget_moves"],
        })
    df = pd.DataFrame(rows).sort_values("Mean Score", ascending=False).reset_index(drop=True)
    df.index = df.index + 1
    df.index.name = "Rank"
    return df

leaderboards = {}
for mode_name, mode_data in modes.items():
    print(f"=== BẢNG XẾP HẠNG CHẾ ĐỘ: {mode_name.upper()} (time_budget={mode_data['time_budget']}s) ===")
    lb = build_leaderboard(mode_data, agents)
    leaderboards[mode_name] = lb
    display(lb.drop(columns=["key", "ci_lo", "ci_hi"]))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6.2), dpi=140)
mode_titles = {
    "time_per_move": "Chế độ 1: Cùng thời gian suy nghĩ (0.5s / nước)",
    "default_config": "Chế độ 2: Cấu hình mặc định (depth=4 / sims=1000)",
}

for ax, (mode_name, mode_data) in zip(axes, modes.items()):
    order = list(leaderboards[mode_name]["key"])
    mat = np.full((len(order), len(order)), np.nan)
    for i, r_ag in enumerate(order):
        for j, c_ag in enumerate(order):
            if i != j:
                mat[i, j] = mode_data["score_matrix"][r_ag][c_ag]

    im = ax.imshow(mat, cmap="RdYlGn", vmin=0.0, vmax=1.0)
    short_labels = [LABELS.get(a, a) for a in order]
    ax.set_xticks(range(len(order)), short_labels, rotation=30, ha="right", fontsize=9.5)
    ax.set_yticks(range(len(order)), short_labels, fontsize=9.5)
    ax.set_title(mode_titles.get(mode_name, mode_name), fontsize=11.5, fontweight="bold", pad=10)
    ax.set_xlabel("Đối thủ (Cột)", fontsize=10)
    ax.set_ylabel("Tác tử (Hàng thắng Cột)", fontsize=10)

    for i in range(len(order)):
        for j in range(len(order)):
            if i == j:
                ax.text(j, i, "—", ha="center", va="center", color="#555555", fontsize=11, fontweight="bold")
            else:
                val = mat[i, j]
                txt_color = "white" if (val < 0.22 or val > 0.82) else "black"
                ax.text(j, i, f"{val:.3f}", ha="center", va="center", color=txt_color, fontsize=9.5, fontweight="bold")

cbar = fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.82, pad=0.03)
cbar.set_label("Score = (Thắng + 0.5 × Hòa) / Số ván", fontsize=10)
fig.suptitle("Ma trận điểm đối đầu trực tiếp giữa 6 cấp độ AI (200 ván/cặp/chế độ)", fontsize=13.5, fontweight="bold", y=1.02)
fig.savefig(FIGS / "01_tournament_score_matrices.png", bbox_inches="tight")
plt.show()
print("Đã lưu hình:", FIGS / "01_tournament_score_matrices.png")

## 2. Chi tiết từng cặp đấu & Trả lời Câu hỏi nghiên cứu 1 và 3

- **Câu hỏi nghiên cứu 1**: *MCTS có thắng Alpha-Beta khi hai bên có cùng thời gian suy nghĩ mỗi nước (0.5s/nước) không? Kết quả có đổi khi đổi sang chế độ cấu hình mặc định không?*
- **Câu hỏi nghiên cứu 3**: *Self-play RL (Cấp 6) có cải thiện tác tử học được so với mô hình bắt chước Alpha-Beta (NN Supervised - Cấp 5) không?*

In [ ]:
pair_rows = []
for mode_name, mode_data in modes.items():
    for m in mode_data["matches"]:
        s = m["a_summary"]
        w_lo, w_hi = s["score_ci"]
        b_lo, b_hi = m["a_score_bootstrap_ci"]
        pair_rows.append({
            "Mode": mode_name,
            "Agent A": LABELS.get(m["a"], m["a"]),
            "Agent B": LABELS.get(m["b"], m["b"]),
            "W - D - L (A)": f"{s['wins']} - {s['draws']} - {s['losses']}",
            "Score A": round(s["score"], 4),
            "Wilson 95% CI": f"[{w_lo:.3f}, {w_hi:.3f}]",
            "Pair-Bootstrap 95% CI": f"[{b_lo:.3f}, {b_hi:.3f}]",
            "Score A (Black)": round(m["a_as_black"]["score"], 3),
            "Score A (White)": round(m["a_as_white"]["score"], 3),
        })

df_pairs = pd.DataFrame(pair_rows)
print("=== KẾT QUẢ TRỌNG TÂM: CÂU HỎI NGHIÊN CỨU 1 (Alpha-Beta vs MCTS) & CÂU HỎI 3 (NN vs RL) ===")
focus = df_pairs[
    ((df_pairs["Agent A"].str.contains("Alpha-Beta")) & (df_pairs["Agent B"].str.contains("MCTS")))
    | ((df_pairs["Agent A"].str.contains("NN Sup")) & (df_pairs["Agent B"].str.contains("RL")))
]
display(focus.reset_index(drop=True))

print("\n=== TOÀN BỘ 15 CẶP ĐẤU Ở 2 CHẾ ĐỘ ===")
display(df_pairs)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14.5, 5.2), dpi=140)

x = np.arange(len(agents))
width = 0.36

for idx, (mode_name, color, label) in enumerate([
    ("time_per_move", "#2b6cb0", "Cùng ngân sách 0.5s/nước (time_per_move)"),
    ("default_config", "#dd6b20", "Cấu hình mặc định (default_config)"),
]):
    lb = leaderboards[mode_name].set_index("key").loc[agents]
    scores = lb["Mean Score"].values
    err_lo = scores - lb["ci_lo"].values
    err_hi = lb["ci_hi"].values - scores
    pos = x + (idx - 0.5) * width
    bars = ax1.bar(pos, scores, width, label=label, color=color, alpha=0.9, edgecolor="black", linewidth=0.6)
    ax1.errorbar(pos, scores, yerr=[err_lo, err_hi], fmt="none", ecolor="#1a202c", capsize=4, lw=1.2)
    for b, sc in zip(bars, scores):
        ax1.text(b.get_x() + b.get_width()/2, sc + 0.03, f"{sc:.2f}", ha="center", va="bottom", fontsize=8.5, fontweight="bold")

ax1.set_xticks(x, [LABELS[a] for a in agents], rotation=20, ha="right")
ax1.set_ylim(0, 1.08)
ax1.axhline(0.5, color="gray", linestyle="--", linewidth=0.9, alpha=0.7)
ax1.set_ylabel("Mean Score (trung bình trước 5 đối thủ)")
ax1.set_title("Điểm trung bình toàn giải & Khoảng tin cậy 95% (Wilson)", fontweight="bold")
ax1.legend(loc="upper left", fontsize=9)
ax1.grid(axis="y", linestyle=":", alpha=0.5)

# Biểu đồ đối đầu trực tiếp cho RQ1 (MCTS vs Alpha-Beta) và RQ3 (RL vs NN)
rq_labels = [
    "RQ1: MCTS vs Alpha-Beta\n(time_per_move = 0.5s)",
    "RQ1: MCTS vs Alpha-Beta\n(default_config)",
    "RQ3: RL vs NN Supervised\n(time_per_move = 0.5s)",
    "RQ3: RL vs NN Supervised\n(default_config)",
]
rq_scores, rq_lo, rq_hi = [], [], []
for mode_name in ("time_per_move", "default_config"):
    for m in modes[mode_name]["matches"]:
        if m["a"] == "alphabeta" and m["b"] == "mcts":
            # Điểm của MCTS trước Alpha-Beta = 1 - score(alphabeta)
            sc = 1.0 - m["a_summary"]["score"]
            ci = m["a_score_bootstrap_ci"]
            rq_scores.append((mode_name, "rq1", sc, 1.0 - ci[1], 1.0 - ci[0]))
        elif m["a"] == "nn" and m["b"] == "rl":
            # Điểm của RL trước NN = 1 - score(nn)
            sc = 1.0 - m["a_summary"]["score"]
            ci = m["a_score_bootstrap_ci"]
            rq_scores.append((mode_name, "rq3", sc, 1.0 - ci[1], 1.0 - ci[0]))

rq_ordered = [
    [x for x in rq_scores if x[0] == "time_per_move" and x[1] == "rq1"][0],
    [x for x in rq_scores if x[0] == "default_config" and x[1] == "rq1"][0],
    [x for x in rq_scores if x[0] == "time_per_move" and x[1] == "rq3"][0],
    [x for x in rq_scores if x[0] == "default_config" and x[1] == "rq3"][0],
]
y_pos = np.arange(len(rq_ordered))
vals = np.array([r[2] for r in rq_ordered])
elos = vals - np.array([r[3] for r in rq_ordered])
ehis = np.array([r[4] for r in rq_ordered]) - vals
colors_rq = ["#3182ce", "#63b3ed", "#38a169", "#68d391"]

bars2 = ax2.barh(y_pos, vals, color=colors_rq, height=0.52, edgecolor="black", linewidth=0.6)
ax2.errorbar(vals, y_pos, xerr=[elos, ehis], fmt="none", ecolor="black", capsize=4, lw=1.3)
ax2.axvline(0.5, color="#e53e3e", linestyle="--", linewidth=1.2, label="Cân bằng (0.50)")
ax2.set_yticks(y_pos, rq_labels, fontsize=9.5)
ax2.invert_yaxis()
ax2.set_xlim(0, 1.08)
ax2.set_xlabel("Score của tác tử đứng trước (kèm Pair-Bootstrap 95% CI)")
ax2.set_title("Đối đầu trực tiếp: Câu hỏi nghiên cứu 1 & 3 (200 ván/cặp)", fontweight="bold")
for b, v, r in zip(bars2, vals, rq_ordered):
    ax2.text(min(v + (r[4]-v) + 0.02, 0.96), b.get_y() + b.get_height()/2, f"{v:.3f} [{r[3]:.2f}, {r[4]:.2f}]", va="center", fontsize=9, fontweight="bold")
ax2.legend(loc="lower right", fontsize=9)
ax2.grid(axis="x", linestyle=":", alpha=0.5)

fig.tight_layout()
fig.savefig(FIGS / "01_mean_scores_and_rq.png", bbox_inches="tight")
plt.show()
print("Đã lưu hình:", FIGS / "01_mean_scores_and_rq.png")

## 3. Chi phí tính toán, Độ trễ (`mean_ms`, `p50_ms`, `p95_ms`) & Bộ nhớ RAM

Tổng hợp thời gian suy nghĩ mỗi nước đi, số nút duyệt (`nodes`) hoặc số mô phỏng (`simulations`) từ log giải đấu, kết hợp với mức tiêu thụ bộ nhớ đỉnh (`peak_tracemalloc_mb`) từ `eval/profile_cost.py`.

In [ ]:
if not PROFILE_PATH.exists():
    from agents.registry import load_agent_specs
    from eval.profile_cost import profile_cost
    specs = load_agent_specs(REPO / "configs" / "agents.yaml")
    prof_data = profile_cost(specs, agents, samples=12, seed=0)
    PROFILE_PATH.write_text(json.dumps(prof_data, ensure_ascii=False, indent=2), encoding="utf-8")

with open(PROFILE_PATH, encoding="utf-8") as f:
    prof = {p["agent"]: p for p in json.load(f)["profiles"]}

def aggregate_agent_cost(mode_data: dict, agent_list: list) -> pd.DataFrame:
    rows = []
    for a in agent_list:
        ms_means, p50s, p95s, maxs, nodes_list, sims_list = [], [], [], [], [], []
        total_moves = 0
        for m in mode_data["matches"]:
            if m["a"] == a or m["b"] == a:
                lat = m["latency"][a]
                cst = m["cost"][a]
                total_moves += lat["moves"]
                ms_means.append(lat["mean_ms"] * lat["moves"])
                p50s.append(lat["p50_ms"])
                p95s.append(lat["p95_ms"])
                maxs.append(lat["max_ms"])
                if cst["nodes_mean"] is not None:
                    nodes_list.append(cst["nodes_mean"])
                if cst["simulations_mean"] is not None:
                    sims_list.append(cst["simulations_mean"])
        ram_mb = prof.get(a, {}).get("peak_tracemalloc_mb")
        rows.append({
            "Agent": LABELS.get(a, a),
            "key": a,
            "Total Moves": total_moves,
            "Mean Latency (ms)": round(sum(ms_means) / total_moves, 3),
            "Median p50 (ms)": round(float(np.mean(p50s)), 3),
            "p95 Latency (ms)": round(float(np.max(p95s)), 3),
            "Max Latency (ms)": round(float(np.max(maxs)), 2),
            "Avg Nodes / Move": round(float(np.mean(nodes_list)), 1) if nodes_list else None,
            "Avg Sims / Move": round(float(np.mean(sims_list)), 1) if sims_list else None,
            "Peak RAM (MB)": round(ram_mb, 3) if ram_mb is not None else None,
            "Over-budget Moves": mode_data["budget"]["by_agent"][a]["over_budget_moves"],
        })
    return pd.DataFrame(rows)

cost_tables = {}
for mode_name, mode_data in modes.items():
    print(f"=== CHI PHÍ TÍNH TOÁN & ĐỘ TRỄ ({mode_name.upper()}) ===")
    ct = aggregate_agent_cost(mode_data, agents)
    cost_tables[mode_name] = ct
    display(ct.drop(columns=["key"]))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14.5, 5.2), dpi=140)

ct_def = cost_tables["default_config"].set_index("key").loc[agents]
ct_tpm = cost_tables["time_per_move"].set_index("key").loc[agents]

x = np.arange(len(agents))
w = 0.36
b1 = ax1.bar(x - w/2, ct_tpm["p95 Latency (ms)"], w, label="p95 ms (time_per_move = 0.5s)", color="#3182ce")
b2 = ax1.bar(x + w/2, ct_def["p95 Latency (ms)"], w, label="p95 ms (default_config)", color="#e53e3e")
ax1.set_yscale("log")
ax1.set_xticks(x, [LABELS[a] for a in agents], rotation=20, ha="right")
ax1.set_ylabel("Độ trễ phân vị 95 - p95 (ms, thang log)")
ax1.set_title("Độ trễ p95 mỗi nước đi của 6 tác tử (Log Scale)", fontweight="bold")
ax1.axhline(500, color="#2d3748", linestyle="--", linewidth=1.0, label="Ngân sách 500 ms (0.5s)")
ax1.legend(fontsize=8.5)
ax1.grid(axis="y", linestyle=":", alpha=0.5)

# Biểu đồ Pareto: Thời gian suy nghĩ trung bình vs Điểm số (default_config)
for a in agents:
    ms_val = ct_def.loc[a, "Mean Latency (ms)"]
    sc_val = modes["default_config"]["mean_score"][a]
    ax2.scatter(ms_val, sc_val, s=130, zorder=4, edgecolors="black")
    ax2.annotate(
        f"{LABELS[a]}\n({ms_val:.2f} ms, score={sc_val:.2f})",
        (ms_val, sc_val),
        textcoords="offset points",
        xytext=(6, -4 if a != "alphabeta" else -18),
        fontsize=8.5,
        fontweight="bold",
    )

ax2.set_xscale("log")
ax2.set_ylim(-0.02, 1.05)
ax2.set_xlim(0.008, 3000)
ax2.set_xlabel("Thời gian suy nghĩ trung bình mỗi nước (ms, thang log)")
ax2.set_ylabel("Mean Score (chế độ default_config)")
ax2.set_title("Đánh đổi Chi phí thời gian vs Sức mạnh tác tử", fontweight="bold")
ax2.grid(True, linestyle=":", alpha=0.6)

fig.tight_layout()
fig.savefig(FIGS / "01_latency_and_cost_tradeoff.png", bbox_inches="tight")
plt.show()
print("Đã lưu hình:", FIGS / "01_latency_and_cost_tradeoff.png")

## 4. Phân tích lợi thế đi trước (Quân Đen vs Quân Trắng)

Vì mỗi cặp khai cuộc đều được chơi 2 ván đổi màu đối xứng, ta có thể kiểm định chính xác xem việc đi trước (Quân Đen) hay đi sau (Quân Trắng) mang lại lợi thế trong giải đấu.

In [ ]:
color_rows = []
for mode_name, mode_data in modes.items():
    by_ag = {a: {"b_w": 0, "b_d": 0, "b_g": 0, "w_w": 0, "w_d": 0, "w_g": 0} for a in agents}
    total_b_wins, total_draws, total_games = 0, 0, 0
    for m in mode_data["matches"]:
        a, b = m["a"], m["b"]
        ab, aw = m["a_as_black"], m["a_as_white"]
        by_ag[a]["b_w"] += ab["wins"]; by_ag[a]["b_d"] += ab["draws"]; by_ag[a]["b_g"] += ab["games"]
        by_ag[a]["w_w"] += aw["wins"]; by_ag[a]["w_d"] += aw["draws"]; by_ag[a]["w_g"] += aw["games"]

        # Khi A cầm Đen thì B cầm Trắng; khi A cầm Trắng thì B cầm Đen
        by_ag[b]["w_w"] += ab["losses"]; by_ag[b]["w_d"] += ab["draws"]; by_ag[b]["w_g"] += ab["games"]
        by_ag[b]["b_w"] += aw["losses"]; by_ag[b]["b_d"] += aw["draws"]; by_ag[b]["b_g"] += aw["games"]

        n_m = m["a_summary"]["games"]
        total_b_wins += round(m["black_win_rate"] * n_m)
        total_draws += round(m["black_draw_rate"] * n_m)
        total_games += n_m

    total_w_wins = total_games - total_b_wins - total_draws
    print(f"[{mode_name}] Tổng {total_games} ván: Đen thắng {total_b_wins} ({100*total_b_wins/total_games:.1f}%), "
          f"Trắng thắng {total_w_wins} ({100*total_w_wins/total_games:.1f}%), Hòa {total_draws} ({100*total_draws/total_games:.1f}%)")

    for a in agents:
        d = by_ag[a]
        sc_b = (d["b_w"] + 0.5 * d["b_d"]) / d["b_g"]
        sc_w = (d["w_w"] + 0.5 * d["w_d"]) / d["w_g"]
        color_rows.append({
            "Mode": mode_name,
            "Agent": LABELS[a],
            "key": a,
            "Score as Black (Đi trước)": round(sc_b, 4),
            "Score as White (Đi sau)": round(sc_w, 4),
            "Diff (Black - White)": round(sc_b - sc_w, 4),
        })

df_color = pd.DataFrame(color_rows)
display(df_color.drop(columns=["key"]))

fig, ax = plt.subplots(figsize=(10, 4.6), dpi=140)
sub = df_color[df_color["Mode"] == "time_per_move"].set_index("key").loc[agents]
x = np.arange(len(agents))
w = 0.35
ax.bar(x - w/2, sub["Score as Black (Đi trước)"], w, label="Cầm quân Đen (Đi trước)", color="#2d3748")
ax.bar(x + w/2, sub["Score as White (Đi sau)"], w, label="Cầm quân Trắng (Đi sau)", color="#edf2f7", edgecolor="#2d3748", linewidth=1.2)
ax.set_xticks(x, [LABELS[a] for a in agents], rotation=15, ha="right")
ax.set_ylim(0, 1.08)
ax.set_ylabel("Score (chế độ time_per_move)")
ax.set_title("So sánh hiệu suất khi cầm quân Đen (đi trước) vs quân Trắng (đi sau)", fontweight="bold")
ax.legend()
ax.grid(axis="y", linestyle=":", alpha=0.5)
fig.tight_layout()
fig.savefig(FIGS / "01_first_player_advantage.png", bbox_inches="tight")
plt.show()
print("Đã lưu hình:", FIGS / "01_first_player_advantage.png")

## 5. Trích xuất ván đấu minh họa từ `games.jsonl` (theo quy tắc định trước)

Để tránh chọn mẫu thiên vị (cherry-picking), ta định trước 2 tiêu chí khách quan để rút ván đấu từ `games.jsonl`:
1. **Ván thua hiếm hoi của tác tử mạnh nhất (`mcts`) trước `alphabeta`** ở chế độ `time_per_move` có cách biệt điểm lớn nhất.
2. **Ván thắng tiêu biểu của `mcts` trước `alphabeta`** trên cùng cặp khai cuộc đổi màu (`pair`) để so sánh đối xứng.

In [ ]:
from othello.encoding import BLACK, WHITE
from othello.game import OthelloGame

ab_mcts_games = []
with open(GAMES_PATH, encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        if rec["mode"] == "time_per_move" and rec["a"] == "alphabeta" and rec["b"] == "mcts":
            ab_mcts_games.append(rec)

# Tìm ván Alpha-Beta thắng MCTS với chênh lệch quân lớn nhất
ab_wins = [r for r in ab_mcts_games if (r["result_black"] == 1 and r["a_is_black"]) or (r["result_black"] == -1 and not r["a_is_black"])]
ab_best_win = max(ab_wins, key=lambda r: abs(r["black_discs"] - r["white_discs"]))
# Tìm ván đối xứng cùng seed & pair (đổi màu)
sister_game = [r for r in ab_mcts_games if r["seed"] == ab_best_win["seed"] and r["pair"] == ab_best_win["pair"] and r["game_id"] != ab_best_win["game_id"]][0]

def replay_disc_diff(rec: dict):
    g = OthelloGame()
    diffs = []
    for a in rec["opening"]:
        g.step(tuple(a))
        b, w = g.score()
        diffs.append(b - w)
    for m in rec["move_log"]:
        g.step(tuple(m["a"]))
        b, w = g.score()
        diffs.append(b - w)
    return diffs, g

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), dpi=140)
for ax, rec, title_prefix in zip(
    axes,
    [ab_best_win, sister_game],
    ["Ván 1 (Alpha-Beta thắng MCTS)", "Ván 2 đối xứng cùng khai cuộc (MCTS thắng lại)"],
):
    diffs, final_g = replay_disc_diff(rec)
    plies = np.arange(1, len(diffs) + 1)
    ax.plot(plies, diffs, color="#2b6cb0", lw=2.0, label="Chênh lệch quân (Đen - Trắng)")
    ax.axhline(0, color="gray", linestyle="--", lw=0.9)
    ax.axvline(len(rec["opening"]), color="#dd6b20", linestyle=":", lw=1.2, label=f"Hết khai cuộc ngẫu nhiên ({len(rec['opening'])} ply)")
    ax.fill_between(plies, diffs, 0, where=(np.array(diffs) >= 0), color="#2d3748", alpha=0.18, label="Đen dẫn điểm")
    ax.fill_between(plies, diffs, 0, where=(np.array(diffs) < 0), color="#e53e3e", alpha=0.18, label="Trắng dẫn điểm")
    ax.set_title(
        f"{title_prefix}\nĐen: {rec['black']} ({rec['black_discs']}) vs Trắng: {rec['white']} ({rec['white_discs']}) | ID: {rec['game_id']}",
        fontsize=9.5,
        fontweight="bold",
    )
    ax.set_xlabel("Nước đi (Ply)")
    ax.set_ylabel("Hiệu số quân (Black − White)")
    ax.legend(fontsize=8, loc="upper left")
    ax.grid(True, linestyle=":", alpha=0.5)

fig.tight_layout()
fig.savefig(FIGS / "01_sample_pair_trajectory.png", bbox_inches="tight")
plt.show()
print("Đã lưu hình:", FIGS / "01_sample_pair_trajectory.png")